# Full Day RFI Flagging Using FRF-Filtered pI SNRs

**by Josh Dillon and Tyler Cox**, last updated September 22, 2026

This notebook brings together the night's delay + fringe-rate-filtered pseudo-Stokes pI SNRs from
[single_baseline_pI_SNR](https://github.com/HERA-Team/hera_notebook_templates/blob/master/notebooks/phase_II/single_baseline_pI_SNR.ipynb)
to make a set of flagging decisions (was H6C's full_day_rfi_round_5). The SNRs are coherently rephased to
each of a catalog of bright sources near transit and averaged over baselines, and also incoherently averaged over all baselines,
and z-scores of each average are flagged iteratively (whole times and channels, outliers, stretches
in time, and a watershed seeded by the previous round's flags). The union of every source's flags
and the incoherent flags is the round's output, one whole-night flag waterfall.
It is the night's final flag waterfall, under which the single-baseline data are then inpainted.

Configuration comes from a TOML file (e.g.
`hera_pipelines/pipelines/phase_II/idr1/v1/analysis/phase_II_analysis.toml`) pointed to by the
`TOML_FILE` environment variable; if none is given, the default settings in the cells below are
used. Environment variables otherwise carry only paths and wrapper-level toggles.

Here's a set of links to skip to particular figures and tables:

• [Figure 1: Waterfalls of pI z-Scores Before Flagging](#Figure-1:-Waterfalls-of-pI-z-Scores-Before-Flagging)

• [Figure 2: Histogram of z-Scores](#Figure-2:-Histogram-of-z-Scores)

• [Figure 3: Waterfalls of pI z-Scores After Flagging](#Figure-3:-Waterfalls-of-pI-z-Scores-After-Flagging)

• [Figure 4: Summary of Flags Before and After Flagging](#Figure-4:-Summary-of-Flags-Before-and-After-Flagging)

In [ ]:
import time
tstart = time.time()
!hostname

In [ ]:
import os
import toml
os.environ['HDF5_USE_FILE_LOCKING'] = 'FALSE'
import h5py
import hdf5plugin  # REQUIRED to have the compression plugins available
import numpy as np
import yaml
import glob
import re
import matplotlib
from scipy.signal import convolve, convolve2d
from pyuvdata import UVFlag
from hera_qm import xrfi
from hera_cal import io, flag_utils
from hera_filters import dspec
import matplotlib.pyplot as plt
from astropy.coordinates import SkyCoord, EarthLocation, AltAz
from astropy.time import Time
import astropy.units as u
from astropy import constants as const

from IPython.display import display, HTML
%matplotlib inline
display(HTML("<style>.container { width:100% !important; }</style>"))
_ = np.seterr(all='ignore')  # get rid of red warnings
%config InlineBackend.figure_format = 'retina'

In [ ]:
# parse wrapper-level environment variables: paths, plus the save toggle
SUM_FILE = os.environ.get("SUM_FILE", None)
# SUM_FILE = '/lustre/aoc/projects/hera/phase-II-analysis/idr1/2459935/zen.2459935.21341.sum.uvh5'
TOML_FILE = os.environ.get("TOML_FILE", None)
# TOML_FILE = '/lustre/aoc/projects/hera/phase-II-analysis/idr1/src/hera_pipelines/pipelines/phase_II/idr1/v1/analysis/phase_II_analysis.toml'
SAVE_RESULTS = os.environ.get("SAVE_RESULTS", "TRUE").upper() == "TRUE"
# which [WorkFlow] action is running this notebook
ACTION = os.environ.get("ACTION", "FULL_DAY_RFI_PI_FRF_NOTEBOOK")

# default names for the files this notebook reads or writes: single-baseline files end in the suffixes, per-night
# products are relative to SUM_FILE's folder ({JD} is the integer JD), and the source catalog is an absolute path
SINGLE_BASELINE_SUFFIX = 'sum.smooth_calibrated.red_avg.uvh5'
PI_FRF_SNR_SUFFIX = 'sum.pI_FRF_SNR.uvh5'
CORNER_TURN_MAP_FILENAME = 'single_baseline_files/corner_turn_map.yaml'
FLAGS_PI_DLYFILT_FILENAME = 'single_baseline_files/zen.{JD}.flag_waterfall_pI_dlyfilt.h5'  # the previous round's flags, which seed the watershed
FLAGS_PI_FRF_FILENAME = 'single_baseline_files/zen.{JD}.flag_waterfall_pI_FRF.h5'
REPHASING_SOURCES_FILENAME = '/lustre/aoc/projects/hera/phase-II-analysis/idr1/src/hera_pipelines/pipelines/phase_II/idr1/v1/analysis/source_catalogs/rephasing_sources.yaml'

# shared across the pipeline via [GLOBAL_OPTS]
BAND_SPLIT_FREQ = 100.0  # in MHz; divides the low and high bands (inside the a priori flagged FM gap)
MAX_FREQ_FLAG_FRAC = 0.25  # channels flagged more than this fraction of a band's unflagged integrations are flagged throughout
MAX_TIME_FLAG_FRAC = 0.5  # integrations flagged in more than this fraction of a band's unflagged channels are flagged across it

# default settings, overridden by the TOML's [FULL_DAY_RFI_PI_FRF_OPTS] section (if given)
MIN_SAMP_FRAC = 0.1  # baselines with fewer median nsamples than this fraction of the autocorrelations' are left out
Z_THRESH = 8.0
WS_Z_THRESH = 4.0
AVG_Z_THRESH = 2.0
TIME_CONV_SIZE = 3600.0  # in seconds; the longest stretch in time tested for a low-level average excess

toml_options = (toml.load(TOML_FILE) if TOML_FILE is not None else {})

# Take the suffixes this notebook reads or writes from [DATA_PRODUCTS], scoped by that
# section's produced_by/consumed_by wiring, and require that wiring to agree exactly with
# the defaults above. That way the arrows drawn on the pipeline flowchart cannot drift from
# what this notebook actually touches: adding an edge there without using the file here (or
# vice versa) fails loudly, in the first cell, rather than silently.
declared_suffixes = {name for name in list(globals()) if name.endswith('_SUFFIX')}
wired_suffixes = set()
for product, spec in toml_options.get('DATA_PRODUCTS', {}).items():
    producers, consumers = (spec.get(key, []) for key in ['produced_by', 'consumed_by'])
    producers = ([producers] if isinstance(producers, str) else producers)
    consumers = ([consumers] if isinstance(consumers, str) else consumers)
    if 'suffix' in spec and (ACTION in producers or ACTION in consumers):
        wired_suffixes.add(f'{product.upper()}_SUFFIX')
        globals()[f'{product.upper()}_SUFFIX'] = spec['suffix']
if toml_options:
    assert wired_suffixes == declared_suffixes, (
        f'[DATA_PRODUCTS] wires {sorted(wired_suffixes)} to {ACTION}, '
        f'but this notebook declares {sorted(declared_suffixes)}.')
# per-night products have no per-file suffix; their [DATA_PRODUCTS] filenames are relative to SUM_FILE's folder
for product in [name[:-len('_FILENAME')] for name in list(globals()) if name.endswith('_FILENAME')]:
    globals()[f'{product}_FILENAME'] = toml_options.get('DATA_PRODUCTS', {}).get(product, {}).get('filename', globals()[f'{product}_FILENAME'])

for toml_section in ['GLOBAL_OPTS', 'FULL_DAY_RFI_PI_FRF_OPTS']:
    if toml_section in toml_options:
        print(f'Loading overrides from [{toml_section}] in {TOML_FILE}.')
        for key, val in toml_options[toml_section].items():
            globals()[key.upper()] = val

jdstr = re.search(r'zen\.(\d+)\.', os.path.basename(SUM_FILE)).group(1)
CORNER_TURN_MAP_YAML = os.path.join(os.path.dirname(SUM_FILE), CORNER_TURN_MAP_FILENAME)
PRIOR_FLAG_FILE = os.path.join(os.path.dirname(SUM_FILE), FLAGS_PI_DLYFILT_FILENAME.format(JD=jdstr))
OUTFILE = os.path.join(os.path.dirname(SUM_FILE), FLAGS_PI_FRF_FILENAME.format(JD=jdstr))

for setting in ['SUM_FILE', 'TOML_FILE', 'SAVE_RESULTS', 'ACTION', 'SINGLE_BASELINE_SUFFIX', 'PI_FRF_SNR_SUFFIX', 'CORNER_TURN_MAP_YAML',
                'PRIOR_FLAG_FILE', 'OUTFILE', 'REPHASING_SOURCES_FILENAME', 'BAND_SPLIT_FREQ', 'MAX_FREQ_FLAG_FRAC', 'MAX_TIME_FLAG_FRAC',
                'MIN_SAMP_FRAC', 'Z_THRESH', 'WS_Z_THRESH', 'AVG_Z_THRESH', 'TIME_CONV_SIZE']:
    print(f'{setting} = {eval(setting)}')

In [ ]:
# the bright sources to rephase to, with how far from transit (in hours) each is used
with open(REPHASING_SOURCES_FILENAME) as f:
    SOURCES = {source['name']: (source['right_ascension'], source['declination'], source['pI_FRF_window_hours'])
               for source in yaml.safe_load(f)['sources']}
print('Rephasing sources and their windows in hours: ' + ', '.join(f'{name} ({width})' for name, (_, _, width) in SOURCES.items()))

## Load data

In [ ]:
with open(CORNER_TURN_MAP_YAML, 'r') as file:
    corner_turn_map = yaml.unsafe_load(file)

In [ ]:
all_outfiles = [outfile for outfiles in corner_turn_map['files_to_outfiles_map'].values() for outfile in outfiles]
all_snr_files = [outfile[:-len(SINGLE_BASELINE_SUFFIX)] + PI_FRF_SNR_SUFFIX for outfile in all_outfiles]
extant_snr_files = [snr_file for snr_file in all_snr_files if os.path.exists(snr_file)]
print(f'Found {len(extant_snr_files)} SNR files, starting with {extant_snr_files[0]}')

In [ ]:
# the averaged autocorrelations' nsamples, as the reference for which baselines have enough samples
for outfile in all_outfiles:
    match = re.search(r'\.(\d+)_(\d+)\.', os.path.basename(outfile))
    if match and match.group(1) == match.group(2):
        hd_autos = io.HERAData(outfile)
        _, _, auto_nsamples = hd_autos.read(polarizations=['ee', 'nn'])
        dt = np.median(np.diff(hd_autos.times))
        break

# For pI data, use the mean of ee and nn auto nsamples as the reference
med_auto_nsamples_pI = np.mean([np.median(auto_nsamples[bl]) for bl in auto_nsamples])
del auto_nsamples

In [ ]:
# This notebook assumes that the phasing is unprojected (i.e. untracked)
cat_types = list(set([cat['cat_type'] for cat in hd_autos.phase_center_catalog.values()]))
assert cat_types == ['unprojected'], "Expected only unprojected phasing, but found: {cat_types}"

In [ ]:
# load up SNRs, counts, and nsamples
SNRs = {}
SNR_counts = {}
SNR_med_nsamples = {}

# also accumulate incoherent |SNR| sum in the same loop
abs_SNR_sum = None  # initialized on first valid baseline
abs_SNR_count = None

for snr_file in extant_snr_files:
    hd = io.HERADataFastReader(snr_file)
    _, _, nsamples = hd.read(read_data=False, read_flags=False)
    med_nsamples_here = np.max([np.median(n) for n in nsamples.values()])
    if not (med_nsamples_here > MIN_SAMP_FRAC * med_auto_nsamples_pI):
        continue
    
    data, flags, _ = hd.read(read_nsamples=False)

    for bl in data:
        
        SNRs[bl] = np.where(flags[bl], 0, data[bl])
        SNR_counts[bl] = np.where(flags[bl], 0, 1)
        SNR_med_nsamples[bl] = np.median(nsamples[bl][~flags[bl]])
        
        # accumulate incoherent sum
        if abs_SNR_sum is None:
            abs_SNR_sum = np.zeros_like(SNRs[bl], dtype=float)
            abs_SNR_count = np.zeros_like(SNR_counts[bl], dtype=float)
        abs_SNR_sum += np.abs(SNRs[bl])  # |complex SNR|, Rayleigh-distributed
        abs_SNR_count += SNR_counts[bl]

print(f"Loaded {len(SNRs)} baselines for rephasing and incoherent combination.")

In [ ]:
# the previous round's flags, which seed the watershed
round4_flags = np.all(UVFlag(PRIOR_FLAG_FILE).flag_array, axis=-1)
print(f'Loaded the prior flags from {PRIOR_FLAG_FILE}: {np.mean(round4_flags):.3%} flagged.')

In [ ]:
# Compute incoherent z-score from |SNR| sum
sigma = 1.0 / np.sqrt(2)  # Rayleigh parameter for |complex noise with unit variance|
rayleigh_mean = sigma * np.sqrt(np.pi / 2)  # = sqrt(pi/4) ~ 0.886
safe_count = np.where(abs_SNR_count > 0, abs_SNR_count, 1)
variance_expected = (4 - np.pi) / 2 * sigma**2 / safe_count
zscore_incoherent = (abs_SNR_sum / safe_count - rayleigh_mean) / variance_expected**.5
zscore_incoherent = np.where(abs_SNR_count == 0, np.nan, zscore_incoherent)

# Recenter per band for robustness
_, (low_band, high_band) = flag_utils.get_minimal_slices(
    ~np.isfinite(zscore_incoherent), freqs=data.freqs,
    freq_cuts=[BAND_SPLIT_FREQ * 1e6])
for band in [low_band, high_band]:
    zscore_incoherent[:, band] -= np.nanmedian(zscore_incoherent[:, band])

# Apply Round 4 flags for watershed seeding
assert round4_flags.shape == (len(hd.times), len(hd.freqs)), \
    f"Round 4 flag shape {round4_flags.shape} doesn't match data shape {(len(hd.times), len(hd.freqs))}"
zscore_incoherent[round4_flags] = np.nan
print(f'Incoherent z-score computed: {np.nanmedian(zscore_incoherent):.3f} median, {np.nanstd(zscore_incoherent):.3f} std.')

## Plotting Functions

In [ ]:
def plot_z_score(zscore, source, source_name, flags=None, vmin=-Z_THRESH, vmax=Z_THRESH, in_range=None, label=None, figsize=(14,6)):
    if in_range is None:
        in_range=slice(None)
    
    if flags is None:
        flags = ~np.isfinite(zscore)
    plt.figure(figsize=figsize, dpi=300)
    extent = [data.freqs[0] / 1e6, data.freqs[-1] / 1e6, 
              data.times[in_range][-1] - int(data.times[0]), data.times[in_range][0] - int(data.times[0])]

    if label is None:
        label=f'pI FRF-Filtered z-score Phased to {source_name} and Coherently Averaged Across Baselines'
    
    plt.imshow(np.where(flags, np.nan, zscore)[in_range, :], aspect='auto', 
               cmap='coolwarm', interpolation='none', vmin=vmin, vmax=vmax, extent=extent)
    plt.colorbar(location='top', extend='both', aspect=40, pad=.02, label=label)
    plt.xlabel('Frequency (MHz)')
    plt.ylabel(f'JD - {int(data.times[0])}')
    plt.tight_layout()
    
    # Add LST right axis with proper wrapping
    lst_grid = hd.lsts[in_range] * 12 / np.pi  # radians to hours
    lst_grid[lst_grid > lst_grid[-1]] -= 24
    ax2 = plt.gca().twinx()
    ax2.set_ylim(lst_grid[-1], lst_grid[0])
    mod24 = lambda x, _: f"{x % 24:.1f}"
    ax2.yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(mod24))
    ax2.set_ylabel('LST (hours)')        
    
    # Mark transit LST on the right axis (only for rephased sources)
    if source is not None:
        transit_lst = source.ra.rad * 12 / np.pi
        # Apply the same 24h wrapping used for the LST grid
        if transit_lst > (hd.lsts[in_range][-1] * 12 / np.pi):
            transit_lst -= 24

        lst_lo, lst_hi = sorted(ax2.get_ylim())
        x_pos = data.freqs[-1] / 1e6  # right edge, next to LST axis

        if lst_lo <= transit_lst <= lst_hi:
            ax2.plot(x_pos, transit_lst, 'ko', markersize=10, clip_on=False,
                     label=f'{source_name} transit')
        elif transit_lst > lst_hi:
            ax2.plot(x_pos, lst_hi, marker='^', color='k', markersize=10,
                     clip_on=False, label=f'{source_name} transit (above)')
        else:
            ax2.plot(x_pos, lst_lo, marker='v', color='k', markersize=10,
                     clip_on=False, label=f'{source_name} transit (below)')
        ax2.legend(loc='upper right')
    
    plt.show()

In [ ]:
def plot_histogram(zscore_per_source, in_range_per_source={}, zscore_incoherent=None):
    plt.figure(figsize=(14,4), dpi=100)
    bins = np.arange(-3, 15, .1)
    
    for source_name, zscore in zscore_per_source.items():
        in_range = in_range_per_source.get(source_name, slice(None))
        hist = plt.hist(np.ravel(zscore[in_range, :]), bins=bins, density=True, label=f'Phased to {source_name}', alpha=.75, histtype='step', zorder=100)
    
    if zscore_incoherent is not None:
        hist = plt.hist(np.ravel(zscore_incoherent), bins=bins, density=True,
                        label='Incoherent Average', alpha=.9, histtype='step',
                        color='k', lw=2, zorder=150)
    
    # Standardized Rayleigh (noise-only reference for rephased z-scores)
    mu_r = np.sqrt(np.pi / 2)
    sig_r = np.sqrt((4 - np.pi) / 2)
    x_r = sig_r * bins + mu_r
    pdf_r = np.where(x_r >= 0,
        sig_r * x_r * np.exp(-0.5 * x_r**2),
        0.0)
    plt.plot(bins, pdf_r, 'k--', label='Standardized Rayleigh (Rephased Noise-Only)', zorder=0)
    
    # Gaussian (noise-only reference for incoherent z-scores, valid by CLT for large N)
    if zscore_incoherent is not None:
        plt.plot(bins, (2*np.pi)**-.5 * np.exp(-bins**2 / 2), 'k:', label='Gaussian (Incoherent Noise-Only)', zorder=0)
    
    plt.axvline(WS_Z_THRESH, c='r', ls='--', label='Watershed z-score')
    plt.axvline(Z_THRESH, c='r', ls='-', label='Threshold z-score')    
    plt.yscale('log')
    all_densities = hist[0][hist[0] > 0]
    if len(all_densities) > 0:
        plt.ylim(np.min(all_densities) / 2, np.max(all_densities) * 2)
    plt.xlim([-3, 15])
    plt.legend()
    plt.xlabel('z-score')
    plt.ylabel('Density')
    plt.tight_layout()

In [ ]:
def summarize_flagging(zscore_per_source, flags_per_source, flags_incoherent=None):

    prior_flags = np.all(~np.isfinite(list(zscore_per_source.values())), axis=0)
    if flags_incoherent is not None:
        prior_flags &= ~np.isfinite(zscore_incoherent)
    
    to_plot = np.zeros(prior_flags.shape, dtype=float)  # no flags
    flag_sum = np.sum(list(flags_per_source.values()), axis=0)
    if flags_incoherent is not None:
        flag_sum_with_incoh = flag_sum + flags_incoherent.astype(int)
    else:
        flag_sum_with_incoh = flag_sum

    n_sources = len(flags_per_source)
    for i, (source_name, flags) in enumerate(flags_per_source.items()):
        to_plot[((flag_sum_with_incoh) == 1) & flags] = i + 2  # flags attributable to a single source
    if flags_incoherent is not None:
        to_plot[((flag_sum_with_incoh) == 1) & flags_incoherent] = n_sources + 2  # incoherent-only flags
    to_plot[flag_sum_with_incoh >= 2] = n_sources + 3  # flags with multiple sources/methods
    to_plot[prior_flags] = 1  # prior flags
    
    plt.figure(figsize=(14,10), dpi=300)
    n_categories = n_sources + 4  # unflagged, prior, per-source..., incoherent, multiple
    if flags_incoherent is None:
        n_categories = n_sources + 3
    colors = list(((0, 0, 0), (.5, .5, .5)) + matplotlib.colormaps["Set2"].colors[0:n_sources])
    if flags_incoherent is not None:
        colors.append(matplotlib.colormaps['Set1'].colors[0])  # red for incoherent
    colors.append((1, 1, 1))  # white for multiple
    cmap = matplotlib.colors.ListedColormap(colors)
    extent = [data.freqs[0] / 1e6, data.freqs[-1] / 1e6, 
              data.times[-1] - int(data.times[0]), data.times[0] - int(data.times[0])]    
    plt.imshow(to_plot,aspect='auto', cmap=cmap, interpolation='none', extent=extent)
    plt.clim([-.5, n_categories - .5])
    cbar = plt.colorbar(location='top', aspect=40, pad=.02)
    cbar.set_ticks(list(range(n_categories)))
    tick_labels = ['Unflagged', 'Previously Flagged'] + [f'Flags from\n{source_name}' for source_name in flags_per_source]
    if flags_incoherent is not None:
        tick_labels.append('Flags from\nIncoherent')
    tick_labels.append('Flags from\nMultiple Sources')
    cbar.set_ticklabels(tick_labels)
    plt.xlabel('Frequency (MHz)')
    plt.ylabel(f'JD - {int(data.times[0])}')

    # Add LST right axis with proper wrapping
    lst_grid = hd.lsts * 12 / np.pi  # radians to hours
    lst_grid[lst_grid > lst_grid[-1]] -= 24
    ax2 = plt.gca().twinx()
    ax2.set_ylim(lst_grid[-1], lst_grid[0])
    mod24 = lambda x, _: f"{x % 24:.1f}"
    ax2.yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(mod24))
    ax2.set_ylabel('LST (hours)')        
    plt.tight_layout()

## Rephasing

In [ ]:
zscore_per_source = {}
in_range_per_source = {}
for source_name in SOURCES:

    print(f'Now computing SNRs rephased to {source_name}.')
    
    # Source and observatory
    source = SkyCoord(*SOURCES[source_name][0:2])
    location = EarthLocation(
        lat=hd.info['latitude'] * u.deg,
        lon=hd.info['longitude'] * u.deg,
        height=hd.info['altitude'] * u.m,
    )    

    # Transform to AltAz at each observation time
    times = Time(hd.times, format='jd')
    source_altaz = source.transform_to(AltAz(obstime=times, location=location))    

    # Build ENU unit vectors from alt/az
    az = source_altaz.az.rad
    alt = source_altaz.alt.rad

    # figure out if any times are within range
    diff = (hd.lsts * 12 / np.pi - source.ra.deg * 24 / 360) % 24
    in_range = np.minimum(diff, 24 - diff) < (SOURCES[source_name][2] / 2)
    if np.sum(in_range) == 0:
        print(f'\t{source_name} is never within {SOURCES[source_name][2]} hours of transit on this JD.')
        continue
    else:
        print(f'\t{np.sum(in_range)} integrations are within {SOURCES[source_name][2]} hours of the transit of {source_name}.')
    
    # rephase visibility by pointing in the direction of the source, minus sign because we're *removing* the delay of the source
    s_enu_over_c = -np.column_stack([np.sin(az) * np.cos(alt), np.cos(az) * np.cos(alt), np.sin(alt)]) / const.c.value
    
    SNR_sum = np.zeros((len(hd.times), len(hd.freqs)), dtype=complex)
    SNR_count = np.zeros((len(hd.times), len(hd.freqs)), dtype=float)
    bls_used = []
    for bl in SNRs:
        if np.median(SNR_med_nsamples[bl]) > MIN_SAMP_FRAC * med_auto_nsamples_pI:
            bl_len = np.linalg.norm(hd.antpos[bl[0]] - hd.antpos[bl[1]])
            if (bl_len > 1):
                bls_used.append(bl)
                bl_vec = hd.antpos[bl[0]] - hd.antpos[bl[1]]  # this is the hera_cal convention that matches utils.lst_rephase
                tau = np.einsum('ti,i->t', s_enu_over_c, bl_vec)
                phs = np.exp(-2j * np.pi * hd.freqs[np.newaxis, :] * tau[:, np.newaxis])  # this also matches the hera_cal convention
                SNR_sum += SNRs[bl] * phs
                SNR_count += SNR_counts[bl]

    sigma = 1.0 / np.sqrt(2)  # Rayleigh parameter for |complex noise with unit variance|
    rayleigh_mean = sigma * np.sqrt(np.pi / 2)  # = √(π/4) ≈ 0.886
    safe_count = np.where(SNR_count > 0, SNR_count, 1)
    predicted_mean = rayleigh_mean / np.sqrt(safe_count)
    variance_expected = (4 - np.pi) / 2 * sigma**2 / safe_count
    zscore = (np.abs(SNR_sum) / safe_count - predicted_mean) / variance_expected**.5
    zscore = np.where(SNR_count == 0, np.nan, zscore)  

    # Apply Round 4 flags for watershed seeding
    assert round4_flags.shape == (len(hd.times), len(hd.freqs)), \
        f"Round 4 flag shape {round4_flags.shape} doesn't match data shape {(len(hd.times), len(hd.freqs))}"
    zscore[round4_flags] = np.nan

    # Skip source if all in-range data is flagged
    if not np.any(np.isfinite(zscore[in_range, :])):
        print(f'\tAll in-range data for {source_name} is flagged. Skipping.')
    else:
        zscore_per_source[source_name] = zscore
        in_range_per_source[source_name] = in_range

## Flagging

In [ ]:
def iteratively_flag_on_averaged_zscore(flags, zscore, avg_func=np.nanmean, avg_z_thresh=AVG_Z_THRESH, verbose=True):
    '''Flag whole integrations or channels based on average z-score. This is done
    iteratively to prevent bad times affecting channel averages or vice versa.'''

    _, (low_band, high_band) = flag_utils.get_minimal_slices(flags, freqs=data.freqs, freq_cuts=[BAND_SPLIT_FREQ * 1e6])
    flagged_chan_count = 0
    flagged_int_count = {low_band: 0, high_band: 0}
    for band in (low_band, high_band):
        while True:
            zspec = avg_func(np.where(flags, np.nan, zscore)[:, band], axis=0)
            ztseries = avg_func(np.where(flags, np.nan, zscore)[:, band], axis=1)
    
            if (np.nanmax(zspec) < avg_z_thresh) and (np.nanmax(ztseries[:]) < avg_z_thresh):
                break
    
            if np.nanmax(zspec) >= np.nanmax(ztseries[:]):
                flagged_chan_count += np.sum((zspec >= np.nanmax(ztseries)) & (zspec >= avg_z_thresh))
                flags[:, band][:, (zspec >= np.nanmax(ztseries)) & (zspec >= avg_z_thresh)] = True
            else:
                flagged_int_count[band] += np.sum((ztseries >= np.nanmax(zspec)) & (ztseries >= avg_z_thresh))
                flags[(ztseries >= np.nanmax(zspec)) & (ztseries >= avg_z_thresh), band] = True

    ztseries_low = avg_func(np.where(flags, np.nan, zscore)[:, low_band], axis=1)
    flags[(ztseries_low > avg_z_thresh) & np.all(flags[:, high_band], axis=1), low_band] = True
    
    if verbose:
        if (flagged_int_count[low_band] > 0) or (flagged_int_count[high_band] > 0) or (flagged_chan_count > 0):
            print(f'\t\tFlagging an additional {flagged_int_count[low_band]} low-band integrations, '
                  f'{flagged_int_count[high_band]} high-band integrations, and {flagged_chan_count} channels.')

def impose_max_chan_flag_frac(flags, max_flag_frac=MAX_FREQ_FLAG_FRAC, verbose=True):
    '''Flag channels already flagged more than max_flag_frac (excluding completely flagged times).'''
    _, (low_band, high_band) = flag_utils.get_minimal_slices(flags, freqs=data.freqs, freq_cuts=[BAND_SPLIT_FREQ * 1e6])
    for band in [low_band, high_band]:
        unflagged_times = ~np.all(flags[:, band], axis=1)
        frequently_flagged_chans =  np.mean(flags[unflagged_times, band], axis=0) >= max_flag_frac
        if verbose:
            flag_diff_count = np.sum(frequently_flagged_chans) - np.sum(np.all(flags[:, band], axis=0))
            if flag_diff_count > 0:
                print(f'\tFlagging {flag_diff_count} channels previously flagged {max_flag_frac:.2%} or more.')        
        flags[:, band][:, frequently_flagged_chans] = True
        
def impose_max_time_flag_frac(flags, max_flag_frac=MAX_TIME_FLAG_FRAC, verbose=True):
    '''Flag times already flagged more than max_flag_frac (excluding completely flagged channels).'''
    _, (low_band, high_band) = flag_utils.get_minimal_slices(flags, freqs=data.freqs, freq_cuts=[BAND_SPLIT_FREQ * 1e6])
    for name, band in zip(['low', 'high'], [low_band, high_band]):
        unflagged_chans = ~np.all(flags[:, band], axis=0)
        frequently_flagged_times =  np.mean(flags[:, band][:, unflagged_chans], axis=1) >= max_flag_frac
        if verbose:
            flag_diff_count = np.sum(frequently_flagged_times) - np.sum(np.all(flags[:, band], axis=1))
            if flag_diff_count > 0:
                print(f'\t\tFlagging {flag_diff_count} {name}-band times previously flagged {max_flag_frac:.2%} or more.')
        flags[frequently_flagged_times, band] = True

def watershed_flag(flags, zscore, ws_z_thresh=WS_Z_THRESH):
    '''Wrapper around xrfi._ws_flag_waterfall to be performed separately above and below FM.'''
    while True:        
        nflags = np.sum(flags)
        _, (low_band, high_band) = flag_utils.get_minimal_slices(flags, freqs=data.freqs, freq_cuts=[BAND_SPLIT_FREQ * 1e6])
        for band in [low_band, high_band]:
            flags[:, band] |= (xrfi._ws_flag_waterfall(zscore[:, band], flags[:, band], ws_z_thresh))
        if np.sum(flags) == nflags:
            break

def iterative_time_conv_flagging(flags, zscore, conv_size=TIME_CONV_SIZE, one_time_thresh=Z_THRESH, full_kernel_thresh=AVG_Z_THRESH):
    '''Looks for streteches of increasing size that fit a decreasing threshold. At conv_size (in seconds), it flags 
    stretches with average z-score above full_kernel_thresh. At one pixel, it uses one_time_thresh.
    In between, it interpolates logarithmically.'''
    dt_s = dt * 3600 * 24
    heights = np.array([int(h) + 1 for h in 2**np.arange(1, np.ceil(np.log2(conv_size / dt_s) + np.finfo(float).eps))])
    
    # Create cuts that get more strict as the kernel gets bigger
    cuts = one_time_thresh * (full_kernel_thresh / one_time_thresh)**((heights - 1) / (conv_size / dt_s - 1))

    for height, cut in zip(heights, cuts):
        result = {}
        kernel = np.ones((int(height), 1), dtype=float)
        mask = ~(np.isnan(zscore) | flags)
        filled_data = np.where(mask, zscore, 0.0)
        conv_data = convolve2d(filled_data, kernel, mode='same')
        conv_mask = convolve2d(mask.astype(float), kernel, mode='same')
        with np.errstate(divide='ignore', invalid='ignore'):
            result = conv_data / conv_mask

        above_cut = result > cut 
        flags[:, :] |= (convolve2d(above_cut.astype(float), kernel, mode='same') > 0)
        
        print(f'\t\t{np.mean(flags):.3%} of waterfall flagged after {height}-integration convolution-based flagging with z-scores above {cut:.3f}.')

In [ ]:
flags_per_source = {}

for source_name, zscore in zscore_per_source.items():
    print(f'Now building flagging mask on SNRs rephased to {source_name}...')
    flags = ~np.isfinite(zscore)
    _, (low_band, high_band) = flag_utils.get_minimal_slices(flags, freqs=data.freqs, freq_cuts=[BAND_SPLIT_FREQ * 1e6])    
    print(f'\t{np.mean(flags):.3%} of waterfall flagged to start.')

    # flag whole integrations or channels using outliers in median
    while True:
        nflags = np.sum(flags)  
        iteratively_flag_on_averaged_zscore(flags, zscore, avg_func=np.nanmedian, avg_z_thresh=AVG_Z_THRESH, verbose=True)
        impose_max_chan_flag_frac(flags, max_flag_frac=MAX_FREQ_FLAG_FRAC, verbose=True)
        impose_max_time_flag_frac(flags, max_flag_frac=MAX_TIME_FLAG_FRAC, verbose=True)
        if np.sum(flags) == nflags:
            break  
    print(f'\t{np.mean(flags):.3%} of waterfall flagged after flagging whole times and channels with median z > {AVG_Z_THRESH}.')
    
    for band in [low_band, high_band]:
        flags[:, band] |= (zscore[:, band] > Z_THRESH) 
    print(f'\t{np.mean(flags):.3%} of waterfall flagged after flagging z > {Z_THRESH} outliers.')

    # iterative time-convolved flagging
    iterative_time_conv_flagging(flags, zscore, conv_size=TIME_CONV_SIZE, one_time_thresh=Z_THRESH, full_kernel_thresh=AVG_Z_THRESH)
    print(f'\t{np.mean(flags):.3%} of waterfall flagged after time convolution flagging.')

    # watershed flagging 
    watershed_flag(flags, zscore, ws_z_thresh=WS_Z_THRESH)
    print(f'\t{np.mean(flags):.3%} of waterfall flagged after watershed flagging on z > {WS_Z_THRESH} neighbors of prior flags (seeded with the prior flags).')

    # flag whole integrations or channels using outliers in mean
    while True:
        nflags = np.sum(flags)
        iteratively_flag_on_averaged_zscore(flags, zscore, avg_func=np.nanmean, avg_z_thresh=AVG_Z_THRESH, verbose=True)
        impose_max_chan_flag_frac(flags, max_flag_frac=MAX_FREQ_FLAG_FRAC, verbose=True)
        impose_max_time_flag_frac(flags, max_flag_frac=MAX_TIME_FLAG_FRAC, verbose=True)
        if np.sum(flags) == nflags:
            break  
    print(f'\t{np.mean(flags):.3%} of waterfall flagged after flagging whole times and channels with average z > {AVG_Z_THRESH}.')
    
    # watershed flagging one last time
    watershed_flag(flags, zscore, ws_z_thresh=WS_Z_THRESH)
    print(f'\t{np.mean(flags):.3%} of waterfall flagged after watershed flagging again on z > {WS_Z_THRESH} neighbors of prior flags.')    

    flags_per_source[source_name] = np.where(in_range_per_source[source_name][:, None], flags, ~np.isfinite(zscore))
    flags_per_source[source_name] = np.where(np.all(flags, axis=0)[None, :], flags, flags_per_source[source_name])
    print('')

## Incoherent Flagging

In [ ]:
print('Now building flagging mask on incoherently combined SNRs...')
flags_incoherent = ~np.isfinite(zscore_incoherent)
_, (low_band, high_band) = flag_utils.get_minimal_slices(flags_incoherent, freqs=data.freqs, freq_cuts=[BAND_SPLIT_FREQ * 1e6])    
print(f'\t{np.mean(flags_incoherent):.3%} of waterfall flagged to start.')

# flag whole integrations or channels using outliers in median
while True:
    nflags = np.sum(flags_incoherent)  
    iteratively_flag_on_averaged_zscore(flags_incoherent, zscore_incoherent, avg_func=np.nanmedian, avg_z_thresh=AVG_Z_THRESH, verbose=True)
    impose_max_chan_flag_frac(flags_incoherent, max_flag_frac=MAX_FREQ_FLAG_FRAC, verbose=True)
    impose_max_time_flag_frac(flags_incoherent, max_flag_frac=MAX_TIME_FLAG_FRAC, verbose=True)
    if np.sum(flags_incoherent) == nflags:
        break  
print(f'\t{np.mean(flags_incoherent):.3%} of waterfall flagged after flagging whole times and channels with median z > {AVG_Z_THRESH}.')

for band in [low_band, high_band]:
    flags_incoherent[:, band] |= (zscore_incoherent[:, band] > Z_THRESH) 
print(f'\t{np.mean(flags_incoherent):.3%} of waterfall flagged after flagging z > {Z_THRESH} outliers.')

# iterative time-convolved flagging
iterative_time_conv_flagging(flags_incoherent, zscore_incoherent, conv_size=TIME_CONV_SIZE, one_time_thresh=Z_THRESH, full_kernel_thresh=AVG_Z_THRESH)
print(f'\t{np.mean(flags_incoherent):.3%} of waterfall flagged after time convolution flagging.')

# watershed flagging 
watershed_flag(flags_incoherent, zscore_incoherent, ws_z_thresh=WS_Z_THRESH)
print(f'\t{np.mean(flags_incoherent):.3%} of waterfall flagged after watershed flagging on z > {WS_Z_THRESH} neighbors of prior flags (seeded with the prior flags).')

# flag whole integrations or channels using outliers in mean
while True:
    nflags = np.sum(flags_incoherent)
    iteratively_flag_on_averaged_zscore(flags_incoherent, zscore_incoherent, avg_func=np.nanmean, avg_z_thresh=AVG_Z_THRESH, verbose=True)
    impose_max_chan_flag_frac(flags_incoherent, max_flag_frac=MAX_FREQ_FLAG_FRAC, verbose=True)
    impose_max_time_flag_frac(flags_incoherent, max_flag_frac=MAX_TIME_FLAG_FRAC, verbose=True)
    if np.sum(flags_incoherent) == nflags:
        break  
print(f'\t{np.mean(flags_incoherent):.3%} of waterfall flagged after flagging whole times and channels with average z > {AVG_Z_THRESH}.')

# watershed flagging one last time
watershed_flag(flags_incoherent, zscore_incoherent, ws_z_thresh=WS_Z_THRESH)
print(f'\t{np.mean(flags_incoherent):.3%} of waterfall flagged after watershed flagging again on z > {WS_Z_THRESH} neighbors of prior flags.')

# Figure 1: Waterfalls of pI z-Scores Before Flagging

In [ ]:
for source_name, zscore in zscore_per_source.items():
    source = SkyCoord(*SOURCES[source_name][0:2])
    plot_z_score(zscore_per_source[source_name], source, source_name, in_range=in_range_per_source[source_name])

# Incoherent z-score waterfall
plot_z_score(zscore_incoherent, None, 'Incoherent',
             label='pI FRF-Filtered z-score Incoherently Averaged Across Baselines', figsize=(14, 10))

# Figure 2: Histogram of z-Scores

In [ ]:
plot_histogram(zscore_per_source, in_range_per_source=in_range_per_source, zscore_incoherent=zscore_incoherent)

# Figure 3: Waterfalls of pI z-Scores After Flagging

In [ ]:
for source_name, zscore in zscore_per_source.items():
    source = SkyCoord(*SOURCES[source_name][0:2])
    plot_z_score(zscore, source, source_name, flags=flags_per_source[source_name], in_range=in_range_per_source[source_name])

# Incoherent z-score waterfall after flagging
plot_z_score(zscore_incoherent, None, 'Incoherent', flags=flags_incoherent,
             label='pI FRF-Filtered z-score Incoherently Averaged Across Baselines', figsize=(14, 10))

# Figure 4: Summary of Flags Before and After Flagging

In [ ]:
summarize_flagging(zscore_per_source, flags_per_source, flags_incoherent=flags_incoherent)

## Save results

In [ ]:
# the round's flags are the union of every source's and the incoherent ones: one whole-night waterfall, the
# product downstream reads (with no source in range, the previous round's flags carry through unchanged). Each source's
# own flags and the incoherent flags are also written, as in H6C, for provenance.
all_flags = dict(flags_per_source); all_flags['incoherent'] = flags_incoherent
flags = (np.any(list(all_flags.values()), axis=0) if len(all_flags) > 0 else round4_flags)
print(f'{np.mean(flags):.3%} of the waterfall is flagged after this round (was {np.mean(round4_flags):.3%}).')
if SAVE_RESULTS:
    uvf = UVFlag(hd_autos, mode='flag', waterfall=True)
    for name, flags_here in list(all_flags.items()) + [(None, flags)]:
        for polind in range(uvf.flag_array.shape[2]):
            uvf.flag_array[:, :, polind] = flags_here
        outfile = (OUTFILE if name is None else OUTFILE.replace(f'zen.{jdstr}.', f'zen.{jdstr}.{name.replace(" ", "_")}.'))
        uvf.write(outfile, clobber=True)
        print(f'Wrote {outfile}')

## Metadata

In [ ]:
for repo in ['hera_cal', 'hera_qm', 'hera_filters', 'hera_notebook_templates', 'pyuvdata']:
    exec(f'from {repo} import __version__')
    print(f'{repo}: {__version__}')

In [ ]:
print(f'Finished execution in {(time.time() - tstart) / 60:.2f} minutes.')